Select three datasets to perform your analysis on. For example, you can use the Adult, German, and COMPAS datasets.

In [69]:
import pandas as pd

adult_df = pd.read_csv('data/adult.csv')
german_df = pd.read_csv('data/GermanCredit.csv')
compas_df = pd.read_csv('data/compas-scores-two-years.csv')


print(adult_df.head(3))
print(german_df.head(3))
print(compas_df.head(3))


   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   

        occupation   relationship   race     sex  capital.gain  capital.loss  \
0                ?  Not-in-family  White  Female             0          4356   
1  Exec-managerial  Not-in-family  White  Female             0          4356   
2                ?      Unmarried  Black  Female             0          4356   

   hours.per.week native.country income  
0              40  United-States  <=50K  
1              18  United-States  <=50K  
2              40  United-States  <=50K  
                status  duration                            credit_history  \
0         ... < 100 DM         6   critical account/other credits existing   
1    0 <= ... < 200 DM        48  existing credits paid back duly 

Compute the demographic parity difference (DP_Diff), the disparate impact ratio (DIR), and the consistency from the original labels of the dataset before mitigation and training any classifiers.

In [70]:
import pandas as pd
from aif360.sklearn.metrics import consistency_score
from fairlearn.metrics import (
    demographic_parity_difference,
    demographic_parity_ratio,
)

# Numerize all categorical columns
for df in [adult_df, german_df, compas_df]:
    for col in df.select_dtypes(include=["object", "category"]).columns:
        df[col] = df[col].astype("category").cat.codes

# Adult
adult_dp_diff = demographic_parity_difference(
    y_true=adult_df["income"],
    y_pred=adult_df["income"],
    sensitive_features=adult_df["sex"],
)
adult_dir_ratio = demographic_parity_ratio(
    y_true=adult_df["income"],
    y_pred=adult_df["income"],
    sensitive_features=adult_df["sex"],
)
adult_consistency = consistency_score(
    adult_df.drop(columns=["income", "sex"]),
    adult_df["income"],
)
print("[Adult] DP_Diff: ", adult_dp_diff)
print("[Adult] DIR: ", adult_dir_ratio)
print("[Adult] Consistency: ", adult_consistency)
print("-" * 20)

# German
german_dp_diff = demographic_parity_difference(
    y_true=german_df["credit_risk"],
    y_pred=german_df["credit_risk"],
    sensitive_features=german_df["age"],
)
german_dir_ratio = demographic_parity_ratio(
    y_true=german_df["credit_risk"],
    y_pred=german_df["credit_risk"],
    sensitive_features=german_df["age"],
)
german_consistency = consistency_score(
    german_df.drop(columns=["credit_risk", "age"]),
    german_df["credit_risk"],
)
print("[German] DP_Diff: ", german_dp_diff)
print("[German] DIR: ", german_dir_ratio)
print("[German] Consistency: ", german_consistency)
print("-" * 20)

# Compas
compas_dp_diff = demographic_parity_difference(
    y_true=compas_df["v_score_text"],
    y_pred=compas_df["v_score_text"],
    sensitive_features=compas_df["race"],
)
compas_dir_ratio = demographic_parity_ratio(
    y_true=compas_df["v_score_text"],
    y_pred=compas_df["v_score_text"],
    sensitive_features=compas_df["race"],
)
compas_consistency = consistency_score(
    compas_df.drop(columns=["v_score_text", "race"]).fillna(0),
    compas_df["v_score_text"],
)
print("[Compas] DP_Diff: ", compas_dp_diff)
print("[Compas] DIR: ", compas_dir_ratio)
print("[Compas] Consistency: ", compas_consistency)

[Adult] DP_Diff:  0.19627598779361352
[Adult] DIR:  0.3580225496813511
[Adult] Consistency:  0.768643469180922
--------------------
[German] DP_Diff:  0.7142857142857143
[German] DIR:  0.2857142857142857
[German] Consistency:  0.6819999999999999
--------------------
[Compas] DP_Diff:  0.29180194805194803
[Compas] DIR:  0.6541606541606542
[Compas] Consistency:  0.6672026614915443


Perform a train/test split with stratification.


In [71]:
from sklearn.model_selection import train_test_split

adult_X = adult_df.drop(columns=["income"])
adult_y = adult_df["income"]

adult_X_train, adult_X_test, adult_y_train, adult_y_test = train_test_split(
    adult_X,
    adult_y,
    test_size=0.20,
    random_state=2026,
    stratify=adult_y,
)

In [72]:
from sklearn.model_selection import train_test_split

german_X = german_df.drop(columns=["credit_risk"])
german_y = german_df["credit_risk"]

german_X_train, german_X_test, german_y_train, german_y_test = train_test_split(
    german_X,
    german_y,
    test_size=0.20,
    random_state=2026,
    stratify=german_y,
)

In [73]:
from sklearn.model_selection import train_test_split

compas_X = compas_df.drop(columns=["v_score_text"])
compas_y = compas_df["v_score_text"]

compas_X_train, compas_X_test, compas_y_train, compas_y_test = train_test_split(
    compas_X,
    compas_y,
    test_size=0.20,
    random_state=2026,
    stratify=compas_y,
)

Follow the appropriate procedure for the selected method: pre-processing methods modify training data, in-processing methods are used during model training, and post-processing methods modify predictions after training.

In [ ]:
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

#  Adult 
adult = pd.read_csv("data/adult.csv")
adult_train, adult_test = train_test_split(adult, test_size=0.2, random_state=2026)
adult_test = adult_test.copy()
adult_test["income"] = adult_test["income"] + "."

adult_train.to_csv("influence-fairness-main/data/adult/adult.data", index=False, header=False)
adult_test.to_csv("influence-fairness-main/data/adult/adult.test", index=False, header=True)

#  German 
german_df["personal_status_sex"] = german_df["personal_status_sex"].apply(
    lambda x: "A92" if str(x).startswith("female") else "A91"
)
german_df["credit_risk"] = german_df["credit_risk"].replace({1: "1", 0: "2"})
for col in german_df.select_dtypes(include="object").columns:
    german_df[col] = german_df[col].str.replace(" ", "_")

german_df.to_csv("influence-fairness-main/data/german/german.data", sep=" ", index=False, header=False)

#  COMPAS 
shutil.copy("data/compas-scores-two-years.csv", "influence-fairness-main/data/compas/compas-scores-two-years.csv")

'influence-fairness-main/data/compas/compas-scores-two-years.csv'

Run these commands in the terminal

```cmd
cd influence-fairness-main
python main.py --dataset adult --metric eop --beta 0.5 --gamma 0.2
python main.py --dataset adult --metric dp --beta 0.5 --gamma 0.2
python main.py --dataset german --metric eop --beta 0.5 --gamma 0.2
python main.py --dataset german --metric dp --beta 0.5 --gamma 0.2
python main.py --dataset compas --metric eop --beta 0.5 --gamma 0.2
python main.py --dataset compas --metric dp --beta 0.5 --gamma 0.2
```


On the held-out test data, report the demographic parity difference (DP_Diff), the disparate impact ratio (DIR), the consistency, and at least one predictive-performance measure from the final predictions after applying the complete mitigation pipeline.



Compare the values of the fairness measures before and after the bias mitigation

### Results are shown in the report